# SQL 1 — Fondamentaux, sur les données ADEME

Ce notebook reprend en SQL l'exploration et le dédoublonnage réalisés en pandas dans les notebooks précédents, sur les données ADEME des véhicules (consommations et émissions de CO₂).

Les requêtes tournent sur **DuckDB** : une base sans serveur, qui lit le CSV directement et dont le SQL est très proche de celui de PostgreSQL. pandas ne sert qu'à afficher les résultats.

Lorsqu'un chiffre a déjà été obtenu en pandas, il est rappelé sous la question (**Référence pandas**) : la requête SQL doit le retrouver.

**Ordre d'exécution d'une requête**, différent de l'ordre d'écriture :

```
FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
```

Il explique pourquoi un alias défini dans le `SELECT` n'existe pas encore dans le `WHERE`, et pourquoi un filtre sur un `count(*)` va dans le `HAVING` (voir 3.2).

## Partie 0 — Charger les données

In [1]:
import duckdb

# Base en mémoire. duckdb.connect("ademe.duckdb") la garderait sur disque.
con = duckdb.connect()


def sql(requete: str):
    """Exécute une requête et renvoie le résultat sous forme de DataFrame."""
    return con.sql(requete).df()


con.execute(
    """
    CREATE OR REPLACE TABLE brut AS
    SELECT * FROM read_csv('../data/vehicules.csv', delim = ';', encoding = 'CP1252')
    """
)
sql("DESCRIBE brut")

,column_name,column_type,null,key,default,extra
0,lib_mrq_doss,VARCHAR,YES,None,None,None
1,lib_mod_doss,VARCHAR,YES,None,None,None
2,mrq_utac,VARCHAR,YES,None,None,None
3,mod_utac,VARCHAR,YES,None,None,None
4,dscom,VARCHAR,YES,None,None,None
5,cnit,VARCHAR,YES,None,None,None
6,tvv,VARCHAR,YES,None,None,None
7,energ,VARCHAR,YES,None,None,None
8,hybride,VARCHAR,YES,None,None,None
9,puiss_admin,BIGINT,YES,None,None,None


**0.1** — Nombre de lignes de la table `brut`.

> Référence pandas : 20 880.

In [2]:
sql("""
    SELECT COUNT(*) FROM brut
""")

,count_star()
0,20880


**0.2** — Nombre de véhicules diesel (`energ = 'GO'`) dans la table `brut`.

> Référence pandas : 12 675.

In [3]:
sql("""
    SELECT COUNT(*) FROM brut
    WHERE TRIM(energ) = 'GO'
""")

,count_star()
0,12675


Un filtre strict `energ = 'GO'` ne retrouve pas ce chiffre : certaines valeurs de `energ` se terminent par des espaces. `TRIM` les retire avant la comparaison. Le même problème était apparu lors de la préparation des données en pandas.

**0.3** — Vue `vehicules` : les colonnes utiles, sous des noms courts, avec les colonnes texte nettoyées par `trim()`. Une vue ne copie aucune donnée : c'est une requête enregistrée sous un nom, réexécutée à chaque usage. **Toutes les requêtes suivantes partent de cette vue.**

| colonne d'origine | nom dans la vue |
|---|---|
| `lib_mrq_doss` | `marque` |
| `lib_mod_doss` | `modele` |
| `energ` | `energ` |
| `typ_boite_nb_rapp` | `boite` |
| `masse_ordma_min` | `masse` |
| `puiss_max` | `puissance` |
| `co2_mixte` | `co2` |

Sur la vue, le comptage diesel du 0.2 n'a plus besoin de `TRIM`.

In [4]:
con.execute("""
    CREATE OR REPLACE VIEW vehicules AS
    SELECT
        trim(lib_mrq_doss) AS marque,
        trim(lib_mod_doss) AS modele,
        trim(energ) AS energ,
        trim(typ_boite_nb_rapp) AS boite,
        masse_ordma_min AS masse,
        puiss_max AS puissance,
        co2_mixte AS co2
    FROM brut
""")

sql("DESCRIBE vehicules")

,column_name,column_type,null,key,default,extra
0,marque,VARCHAR,YES,None,None,None
1,modele,VARCHAR,YES,None,None,None
2,energ,VARCHAR,YES,None,None,None
3,boite,VARCHAR,YES,None,None,None
4,masse,BIGINT,YES,None,None,None
5,puissance,DOUBLE,YES,None,None,None
6,co2,BIGINT,YES,None,None,None


In [5]:
sql("""
    SELECT COUNT(*) FROM vehicules
    WHERE energ = 'GO'
""")

,count_star()
0,12675


## Partie 1 — Sélectionner, filtrer, trier

**1.1** — Les 10 véhicules les plus puissants : marque, modèle, puissance et CO₂.

In [6]:
sql("""
    SELECT marque, modele, puissance, co2 FROM vehicules
    ORDER BY puissance DESC
    LIMIT 10
""")

,marque,modele,puissance,co2
0,FERRARI,LAFERRARI,585.0,330
1,FERRARI,LAFERRARI,585.0,330
2,FERRARI,LAFERRARI,585.0,330
3,FERRARI,LAFERRARI,585.0,330
4,LAMBORGHINI,AVENTADOR,545.0,398
5,LAMBORGHINI,AVENTADOR,545.0,398
6,LAMBORGHINI,AVENTADOR,545.0,398
7,FERRARI,F12,541.0,350
8,FERRARI,F12,541.0,380
9,FERRARI,F12,541.0,380


Plusieurs lignes sont identiques : une ligne du fichier est une variante d'homologation, pas un véhicule distinct. La partie 4 en tire les conséquences.

**1.2** — Nombre de véhicules essence de plus de 300 kW.

In [7]:
sql("""
    SELECT COUNT(*) FROM vehicules
    WHERE energ = 'ES' AND puissance > 300
""")

,count_star()
0,1439


**1.3** — Nombre de marques distinctes, puis leur liste par ordre alphabétique.

In [8]:
sql("""
    SELECT COUNT(DISTINCT marque) FROM vehicules
""")

,count(DISTINCT marque)
0,46


In [9]:
sql("""
    SELECT DISTINCT marque
    FROM vehicules
    ORDER BY marque ASC
""")

,marque
0,ALFA ROMEO
1,AUDI
2,BENTLEY
3,BMW
4,BMW I
5,CADILLAC
6,CHEVROLET
7,CITROEN
8,DACIA
9,DANGEL


## Partie 2 — Les valeurs manquantes

**2.1** — Nombre de lignes, de masses renseignées et de masses manquantes, en une seule requête.

In [10]:
sql("""
        SELECT COUNT(*) AS nb_lignes,
        COUNT(masse) AS nb_masse,
        COUNT(*) - COUNT(masse) AS masse_null
        FROM vehicules
    """)

,nb_lignes,nb_masse,masse_null
0,20880,20880,0


`COUNT(*)` compte toutes les lignes, y compris celles qui contiennent des NULL ; `COUNT(masse)` ne compte que les lignes dont la masse est renseignée. Équivalents pandas : `len(df)` et `df['masse'].count()`.

**2.2** — `WHERE masse = NULL` comparé à `WHERE masse IS NULL`.

In [11]:
sql("""
    SELECT COUNT(*) AS masse_null FROM vehicules
    WHERE masse = NULL
""")

,masse_null
0,0


`WHERE masse = NULL` renvoie 0 : en SQL, NULL signifie « valeur inconnue », et toute comparaison avec une valeur inconnue donne elle-même NULL, jamais vrai. Or le `WHERE` ne garde que les lignes où la condition est vraie. Le résultat serait 0 **même s'il y avait des masses manquantes** ; c'est pourquoi on utilise `IS NULL`.

In [12]:
sql("""
    SELECT COUNT(*) AS masse_is_null FROM vehicules
    WHERE masse IS NULL
""")

,masse_is_null
0,0


La masse n'ayant aucune valeur manquante, le même test sur `co2` est plus parlant :

In [13]:
sql("""
    SELECT COUNT(*) AS co2_null FROM vehicules
    WHERE co2 = NULL
""")

,co2_null
0,0


In [14]:
sql("""
    SELECT COUNT(*) AS co2_is_null FROM vehicules
    WHERE co2 IS NULL
""")

,co2_is_null
0,56


`= NULL` donne toujours 0, alors que `IS NULL` trouve les 56 CO₂ manquants, ceux des véhicules électriques (`EL`).

## Partie 3 — Agréger

**3.1** — Par motorisation : nombre de véhicules, CO₂ moyen arrondi à une décimale, minimum et maximum, du plus fréquent au moins fréquent.

> Référence pandas : GO 12 675 véhicules et 148,4 g/km ; ES 6 934 et 171,0 ; GH 110,9.

In [15]:
sql("""
    SELECT energ, COUNT(*) AS nb_vehicules,
    ROUND(AVG(co2), 1) AS moyenne_co2,
    MIN(co2) AS min_co2,
    MAX (co2) AS max_co2
    FROM vehicules
    GROUP BY energ
    ORDER BY nb_vehicules DESC
""")

,energ,nb_vehicules,moyenne_co2,min_co2,max_co2
0,GO,12675,148.4,79,295
1,ES,6934,171.0,84,398
2,EH,546,134.8,75,330
3,GH,386,110.9,85,169
4,EE,90,58.4,13,79
5,GN,61,180.4,79,226
6,EL,56,NaN,<NA>,<NA>
7,ES/GN,52,186.0,107,286
8,GN/ES,36,114.6,86,134
9,GP/ES,21,128.3,106,179


Le CO₂ moyen des électriques (`EL`) n'existe pas : leurs valeurs sont manquantes dans le fichier, et non égales à 0. `AVG` ignore les valeurs manquantes, n'a donc rien à moyenner et renvoie NULL, affiché NaN par pandas. C'est le comportement de `mean()` en pandas, qui ignore les NaN par défaut et renvoie NaN quand il n'y a que des NaN.

**3.2** — Marques comptant au moins 50 véhicules essence, avec leur effectif et leur CO₂ moyen, de la plus émettrice à la moins émettrice.

In [16]:
sql("""
    SELECT marque,
    COUNT(*) AS nb_vehicules,
    ROUND(AVG(co2), 1) AS moyenne_co2
    FROM vehicules
    WHERE energ = 'ES'
    GROUP BY marque
    HAVING COUNT(*) >= 50
    ORDER BY moyenne_co2 DESC
""")

,marque,nb_vehicules,moyenne_co2
0,JAGUAR LAND ROVER LIMITED,52,254.6
1,LEXUS,230,237.1
2,MERCEDES AMG,283,227.2
3,PORSCHE,233,219.1
4,MERCEDES BENZ,3815,167.4
5,BMW,533,163.0
6,AUDI,204,154.5
7,MINI,90,142.4
8,ALFA ROMEO,108,139.3
9,OPEL,123,137.4


La condition « au moins 50 » ne peut pas aller dans le `WHERE` : le `WHERE` s'exécute avant le `GROUP BY` et examine les lignes une par une, il ne peut donc pas savoir combien de véhicules essence compte chaque marque. Le `HAVING` s'exécute après le regroupement et filtre les groupes, une fois les `COUNT` calculés.

**3.3** — Véhicules essence par tranche de puissance, avec `CASE WHEN` : jusqu'à 75 kW, de 75 à 150, de 150 à 300, au-delà de 300. Effectif et CO₂ moyen par tranche, dans l'ordre des puissances.

In [17]:
sql("""
    SELECT
        CASE
            WHEN puissance <= 75 THEN '0-75 kW' 
            WHEN puissance <= 150 THEN '75-150 kW'
            WHEN puissance <= 300 THEN '150-300 kW'
            WHEN puissance > 300 THEN '> à 300 kW'
            ELSE 'inconnue'
        END AS tranche_puissance,
        COUNT(*) AS effectif, ROUND(AVG(co2), 1) AS moyenne_co2
    FROM vehicules
    WHERE energ = 'ES'
    GROUP BY tranche_puissance
    ORDER BY AVG(puissance)
""")

,tranche_puissance,effectif,moyenne_co2
0,0-75 kW,534,115.0
1,75-150 kW,2762,136.7
2,150-300 kW,2199,177.0
3,> à 300 kW,1439,248.2


Un `ORDER BY tranche_puissance` ne donne pas le bon ordre : il trie du texte, caractère par caractère (« 150… » passe avant « 75… »). Le tri se fait donc sur la puissance moyenne de chaque tranche, qui suit l'ordre des tranches par construction puisqu'elles ne se chevauchent pas.

**3.4** — Par motorisation, part (en %) des véhicules soumis au malus de 2014, c'est-à-dire émettant 131 g/km ou plus.

La moyenne d'une colonne de 0 et de 1 est une proportion. Sans `ELSE`, le `CASE` laisse à NULL les CO₂ inconnus, que `AVG` ignore.

In [18]:
sql("""
    SELECT energ, COUNT(co2) AS nb_co2, ROUND(AVG(
        CASE
            WHEN co2 >= 131 THEN 1
            WHEN co2 < 131 THEN 0
        END) * 100, 2) AS pct_malus
    FROM vehicules
    GROUP BY energ
    ORDER BY nb_co2
""")

,energ,nb_co2,pct_malus
0,EL,0,NaN
1,GL,1,0.00
2,FE,1,100.00
3,GP/ES,21,52.38
4,ES/GP,21,80.95
5,GN/ES,36,13.89
6,ES/GN,52,88.46
7,GN,61,65.57
8,EE,90,0.00
9,GH,386,2.07


`nb_co2` indique sur combien de véhicules repose chaque pourcentage : aucun pour `EL` (CO₂ non renseigné), un seul pour `FE` et `GL`, dont les 100 % et 0 % ne sont donc pas significatifs. `ES` et `GO`, eux, reposent sur des milliers de véhicules.

## Partie 4 — Le dédoublonnage du notebook 02, en SQL

Le dédoublonnage de `preparer()`, réécrit en SQL. Rappel : une ligne du fichier est une variante d'homologation, pas un véhicule.

**4.1** — Lignes essence exploitables, c'est-à-dire avec une masse, une puissance et un CO₂ renseignés.

In [19]:
sql("""
    SELECT COUNT(*) AS lignes_es
    FROM vehicules
    WHERE energ = 'ES' 
        AND masse IS NOT NULL 
        AND puissance IS NOT NULL 
        AND co2 IS NOT NULL
""")

,lignes_es
0,6934


**4.2** — Parmi elles, véhicules distincts sur la clé (marque, modèle, masse, puissance, CO₂).

> Référence pandas : 2 406.

In [20]:
sql("""
    SELECT COUNT(*) AS vehicules_distincts
    FROM (
        SELECT DISTINCT marque, modele, masse, puissance, co2 
        FROM vehicules
        WHERE energ = 'ES' 
            AND masse IS NOT NULL 
            AND puissance IS NOT NULL 
            AND co2 IS NOT NULL
    ) AS combinaisons
""")

,vehicules_distincts
0,2406


**4.3** — Taux de réplicats, en une seule requête : part des lignes exploitables qui sont des variantes d'un véhicule déjà compté.

> Référence pandas : environ 65 %.

In [21]:
sql("""
    SELECT ROUND((1 - (COUNT(*) / SUM(lignes_exp))) * 100, 2) AS taux_de_replicats
    FROM (
        SELECT marque, modele, masse, puissance, co2, COUNT(*) AS lignes_exp
        FROM vehicules
        WHERE energ = 'ES' 
            AND masse IS NOT NULL 
            AND puissance IS NOT NULL 
            AND co2 IS NOT NULL
        GROUP BY marque, modele, masse, puissance, co2
    ) AS vehicules_distincts
""")

,taux_de_replicats
0,65.3


**4.4** — Corrélation entre masse et puissance (`corr()`) sur les véhicules dédoublonnés, pour l'essence et le diesel, puis sur les lignes brutes.

> Référence pandas : 0,782 pour l'essence, 0,422 pour le diesel.

In [22]:
sql("""
    SELECT energ, corr(masse, puissance) AS corr_masse_puissance
    FROM (
        SELECT DISTINCT energ, marque, modele, masse, puissance, co2
        FROM vehicules
        WHERE energ IN ('ES', 'GO')
            AND masse IS NOT NULL 
            AND puissance IS NOT NULL 
            AND co2 IS NOT NULL
    ) AS combinaisons
    GROUP BY energ
    ORDER BY energ
""")

,energ,corr_masse_puissance
0,ES,0.782109
1,GO,0.421645


In [23]:
sql("""
    SELECT energ, corr(masse, puissance) AS corr_masse_puissance
    FROM vehicules
    WHERE energ IN ('ES', 'GO')
        AND masse IS NOT NULL 
        AND puissance IS NOT NULL 
        AND co2 IS NOT NULL
    GROUP BY energ
    ORDER BY energ
""")

,energ,corr_masse_puissance
0,ES,0.770936
1,GO,0.218659


Sans dédoublonnage, l'écart est faible pour l'essence (0,782 → 0,771), mais la corrélation est divisée par deux pour le diesel (0,422 → 0,219). Sur les lignes brutes, chaque véhicule pèse autant que son nombre de variantes : si les véhicules les plus répliqués s'écartent de la tendance générale, ils la tirent vers eux. Hypothèse pour le diesel : des utilitaires, lourds pour une puissance modeste et déclinés en dizaines de variantes. Vérification :

In [24]:
sql("""
    SELECT marque, modele, masse, puissance, co2, count(*) AS variantes
    FROM vehicules
    WHERE energ = 'GO'
    GROUP BY ALL
    ORDER BY variantes DESC
    LIMIT 15
""")

,marque,modele,masse,puissance,co2,variantes
0,MERCEDES BENZ,VITO,1976,120.0,149,162
1,MERCEDES BENZ,VITO,1976,100.0,152,132
2,MERCEDES BENZ,VITO,1976,65.0,169,81
3,MERCEDES BENZ,VITO,1976,84.0,169,81
4,MERCEDES BENZ,CLASSE V,2050,120.0,149,72
5,MERCEDES BENZ,CLASSE E,1735,100.0,125,72
6,MERCEDES BENZ,CLASSE V,2100,120.0,152,72
7,MERCEDES BENZ,CLASSE V,2075,120.0,149,72
8,MERCEDES BENZ,VITO,1976,100.0,164,66
9,MERCEDES BENZ,VITO,1976,100.0,159,66


Le Vito, modèle le plus répliqué (lignes 0 à 3), garde la même masse (1 976 kg) pour des puissances de 65 à 120 kW : l'inverse de « plus lourd = plus puissant ». Le Classe V suit le même schéma. Sur les lignes brutes, ces combinaisons comptent jusqu'à 162 fois chacune, ce qui fait chuter la corrélation. Ce sont des utilitaires et des monospaces, ce qui confirme l'hypothèse.

**4.5** — Parmi les 2 406 véhicules essence, nombre de configurations distinctes (masse, puissance, boîte).

In [25]:
sql("""
    SELECT COUNT(*) AS configurations_distinctes
    FROM (
        SELECT DISTINCT masse, puissance, boite
        FROM vehicules
        WHERE energ = 'ES' 
            AND masse IS NOT NULL 
            AND puissance IS NOT NULL 
            AND co2 IS NOT NULL
    ) AS combinaisons
""")

,configurations_distinctes
0,1449


L'écart (2 406 − 1 449 = 957) correspond aux véhicules dont la configuration (masse, puissance, boîte) est déjà prise par un autre véhicule, d'une autre marque ou d'un autre modèle. Dans le notebook 04, ces configurations partagées se retrouvaient à la fois dans les données d'entraînement et de validation : une fuite de données, qui rend l'évaluation trop optimiste. Elle a été corrigée par un découpage par groupe sur la configuration (`GroupShuffleSplit` et `GroupKFold`).

## Prochaine étape

SQL 2 : les jointures, les CTE (`WITH`) et les fonctions de fenêtre — `ROW_NUMBER()` pour réécrire le dédoublonnage en gardant une ligne choisie, le « top N par groupe », l'écart de chaque véhicule à la moyenne de sa marque.